In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
df_devices = pd.read_csv('../data/tablas-actualizadas/devices.csv')
df_notifications = pd.read_csv('../data/tablas-actualizadas/notifications.csv')
df_transactions = pd.read_csv('../data/tablas-actualizadas/transactions.csv')
df_users = pd.read_csv('../data/tablas-actualizadas/users.csv')

display(df_devices.head(5), df_notifications.head(5), df_transactions.head(5), df_users.head(5))

In [ ]:
## cambiar el nombre a la tabla

df_devices = df_devices.rename(columns={"string_field_0": "brand_device", "string_field_1": "user_id"})
df_notifications = df_notifications.rename(columns={'created_date': 'create_date_notification'})
df_transactions = df_transactions.rename(columns={'created_date': 'create_date_transaction'})
df_users = df_users.rename(columns={'created_date': 'create_date_user'})

display(df_devices, df_notifications, df_transactions, df_users)

In [ ]:
for df in [df_users, df_devices, df_notifications, df_transactions]:
    if 'Unnamed: 0' in df.columns:
        df.drop(columns='Unnamed: 0', inplace=True)

display(df_devices, df_notifications, df_transactions, df_users)

In [ ]:
# Empieza con df_users
df = df_users.copy()

# Merge con devices
df = df.merge(df_devices, on='user_id', how='left')
df['has_device'] = df['brand_device'].notna()

# Merge con notifications
df = df.merge(df_notifications, on='user_id', how='left')
df['has_notification'] = df['create_date_notification'].notna()

# Merge con transactions
df = df.merge(df_transactions, on='user_id', how='left')
df['has_transaction'] = df['transaction_id'].notna()


In [ ]:
df

In [ ]:
# 19038195

# df = pd.read_csv('../data/df_.csv')

# df = df.drop(columns=['Unnamed: 0'])

# df

In [ ]:
df.columns

In [ ]:
# Asegúrate de que las fechas estén en datetime
df_users['create_date_user'] = pd.to_datetime(df_users['create_date_user'], errors='coerce')
df_notifications['create_date_notification'] = pd.to_datetime(df_notifications['create_date_notification'], errors='coerce')
df_transactions['create_date_transaction'] = pd.to_datetime(df_transactions['create_date_transaction'], errors='coerce')

# Agrupa notifications: primera notificación por user
notifications_agg = (
    df_notifications
    .groupby('user_id', as_index=False)
    .agg(first_notification_date=('create_date_notification', 'min'))
)

# Agrupa transactions: primera y última transacción por user
transactions_agg = (
    df_transactions
    .groupby('user_id', as_index=False)
    .agg(
        first_transaction_date=('create_date_transaction', 'min'),
        last_transaction_date=('create_date_transaction', 'max')
    )
)

# Obtiene la última fecha global de transacción
global_last_txn_date = df_transactions['create_date_transaction'].max()

# Merge users + notifications + transactions
df_merged = (
    df_users
    .merge(notifications_agg, on='user_id', how='left')
    .merge(transactions_agg, on='user_id', how='left')
)

# Flags y métricas
df_merged['has_transaction'] = df_merged['first_transaction_date'].notna()

df_merged['dias_a_primera_txn'] = (
    (df_merged['first_transaction_date'] - df_merged['create_date_user'])
    .dt.days
)

df_merged['converted'] = (
    df_merged['first_notification_date'].notna() &
    df_merged['first_transaction_date'].notna()
)

df_merged['churned'] = (
    df_merged['last_transaction_date'].notna() &
    ((global_last_txn_date - df_merged['last_transaction_date']).dt.days > 60)
)

# Resultado final
df_resumen_general = df_merged.copy()
df_resumen_general.head()


In [ ]:
df_resumen_general

# total de usuarios

In [ ]:

df_resumen_general['user_id'].nunique()

# usuarios con transacciones

In [ ]:
df_resumen_general[df_resumen_general['has_transaction']]['user_id'].nunique()


In [ ]:
conversion_rate = (
    df_resumen_general[df_resumen_general['converted']]['user_id'].nunique() / df_resumen_general['user_id'].nunique()
) * 100
conversion_rate

# AVG de días a convertidos

In [ ]:
avg_days_to_first_txn_ = df_resumen_general['dias_a_primera_txn'].dropna().mean()
avg_days_to_first_txn_

# Churn rate

In [ ]:
total_users = df_resumen_general['user_id'].nunique()
churned_users = df_resumen_general[(df_resumen_general['churned'] == True) & (df_resumen_general['has_transaction'] == True)]['user_id'].nunique()
churn_rate_ = (churned_users / total_users) * 100
churn_rate_

# Evolucion de usuarios

In [ ]:
# Calcular usuarios activos por semana
df_resumen_general['create_date_transaction'] = pd.to_datetime(df_resumen_general['first_transaction_date'], errors='coerce')
df_resumen_general['semana_transaccion'] = df_resumen_general['create_date_transaction'].dt.to_period('W').apply(
    lambda r: r.start_time if pd.notnull(r) else pd.NaT
)
usuarios_activos = (
    df_resumen_general.dropna(subset=['semana_transaccion'])
    .groupby('semana_transaccion')['user_id']
    .nunique()
    .reset_index(name='usuarios_activos')
)

# Calcular usuarios nuevos por semana
df_resumen_general['create_date_user'] = pd.to_datetime(df_resumen_general['create_date_user'], errors='coerce')
df_resumen_general['semana_creacion'] = df_resumen_general['create_date_user'].dt.to_period('W').apply(
    lambda r: r.start_time if pd.notnull(r) else pd.NaT
)
usuarios_nuevos = (
    df_resumen_general.dropna(subset=['semana_creacion'])
    .groupby('semana_creacion')['user_id']
    .nunique()
    .reset_index(name='usuarios_nuevos')
)

df_evolucion = pd.merge(
    usuarios_nuevos,
    usuarios_activos,
    left_on='semana_creacion',
    right_on='semana_transaccion',
    how='outer'
)
df_evolucion['semana'] = df_evolucion['semana_creacion'].combine_first(df_evolucion['semana_transaccion'])
df_evolucion = df_evolucion.dropna(subset=['semana']).sort_values('semana')

fig_evoluacion = px.line(
    df_evolucion,
    x='semana',
    y=['usuarios_nuevos', 'usuarios_activos'],
    labels={'value': 'Cantidad de usuarios', 'variable': 'Tipo de usuario', 'semana': 'Semana'},
)
fig_evoluacion.update_layout(xaxis=dict(tickformat='%Y-%m-%d'))
fig_evoluacion.show()

Los usuarios activos siguen un patrón estable con picos y valles normales, pero mantienen valores consistentes hasta el final del periodo, lo que indica datos completos y coherentes: el comportamiento esperado de una base de datos sin huecos.

In [ ]:
df_transactions

# Curva de retención por semana (cohorte)

In [ ]:
# 1) Copias para no modificar los originales
users = df_users.copy()
transactions = df_transactions.copy()

# 2) Convertir fechas a datetime
users['create_date_user'] = pd.to_datetime(users['create_date_user'], errors='coerce')
transactions['create_date_transaction'] = pd.to_datetime(transactions['create_date_transaction'], errors='coerce')

# 3) Filtrar usuarios con al menos una transacción
usuarios_con_txn = users[users['user_id'].isin(transactions['user_id'].unique())].copy()

# 4) Calcular cohorte: semana de registro del usuario
usuarios_con_txn['cohort_week'] = usuarios_con_txn['create_date_user'].dt.to_period('W').dt.start_time

# 5) Calcular semana de cada transacción
transactions['transaction_week'] = transactions['create_date_transaction'].dt.to_period('W').dt.start_time

# 6) Merge: transacciones + cohorte de cada usuario
merged = transactions.merge(
    usuarios_con_txn[['user_id', 'cohort_week']],
    on='user_id',
    how='inner'  # Aquí clave: solo transacciones de usuarios con cohorte calculada
)

# 7) Calcular semanas desde el registro hasta la transacción
merged['weeks_since_signup'] = (merged['transaction_week'] - merged['cohort_week']).dt.days // 7

# 8) Agrupar: cohorte + semanas desde registro -> usuarios activos únicos
df_retencion = (
    merged
    .groupby(['cohort_week', 'weeks_since_signup'])['user_id']
    .nunique()
    .reset_index(name='active_users')
    .sort_values(['cohort_week', 'weeks_since_signup'])
)

# Mostrar resultado
df_retencion.head(), df_retencion.shape, df_retencion['weeks_since_signup'].max()


In [ ]:
print(df_retencion['cohort_week'].unique())
print(df_retencion['weeks_since_signup'].unique())

In [ ]:
# Filtrar outliers
# df_retencion = df_retencion[(df_retencion['weeks_since_signup'] >= 0) & (df_retencion['weeks_since_signup'] <= 52)]

# Pivot para matriz de retención
retention_pivot = df_retencion.pivot(
    index='cohort_week',
    columns='weeks_since_signup',
    values='active_users'
).fillna(0)

# Normalizar: tasa de retención por cohorte
retention_rate = retention_pivot.divide(retention_pivot.iloc[:, 0], axis=0)

# Plot
fig_matrix = px.imshow(
    retention_rate,
    labels=dict(x='Semanas desde registro', y='Cohorte de registro', color='Tasa de retención'),
    color_continuous_scale='Agsunset'
)
fig_matrix.show()


# Histograma de días a primera transacción

In [ ]:
# Filtrar usuarios con al menos una transacción:
df_txn = df.dropna(subset=['create_date_transaction'])
df_txn

In [ ]:
# Obtener la primera transacción por usuario:
df_first_txn = df_txn.groupby('user_id')['create_date_transaction'].min().reset_index()
df_first_txn.columns = ['user_id', 'first_transaction_date']
df_first_txn

In [ ]:
# Traer fecha de registro:
df_user_creation = df[['user_id', 'create_date_user']].drop_duplicates()
df_merged_dias = pd.merge(df_user_creation, df_first_txn, on='user_id', how='left')
display(df_user_creation, df_merged_dias)

In [ ]:
# Asegura que ambas fechas no tengan zona horaria (tz-naive)
df_merged_dias['first_transaction_date'] = pd.to_datetime(df_merged_dias['first_transaction_date']).dt.tz_localize(None)
df_merged_dias['create_date_user'] = pd.to_datetime(df_merged_dias['create_date_user']).dt.tz_localize(None)

# Calcular días a la primera transacción
df_merged_dias['dias_a_primera_txn'] = (
    df_merged_dias['first_transaction_date'] - df_merged_dias['create_date_user']
).dt.days

df_merged_dias


In [ ]:
# Dibujar el histograma:

fig = px.histogram(
    df_merged_dias,
    x='dias_a_primera_txn',
    nbins=30,
    title='Días hasta la primera transacción',
    labels={'dias_a_primera_txn': 'Días'},
    color_discrete_sequence=['#3BAEDA']
)
fig.show()


La mayoría de los usuarios convierten rápido:

El primer bloque (días cercanos a 0) tiene la mayor altura → eso indica que muchos usuarios hacen su primera transacción en los primeros días después de registrarse.

La conversión se reduce con el tiempo:

A medida que avanzan los días, las barras van decreciendo.

Esto sugiere que cuanto más tiempo pasa sin que el usuario transaccione, menos probable es que lo haga.

Cola larga:

Aunque la mayoría convierte en las primeras semanas, hay usuarios que tardan más de 300 días en hacer su primera transacción (pero son pocos).

# % de usuarios que convierten en 1, 7 y 30 días

In [ ]:
# Filtrar usuarios que sí hicieron transacción (días no nulos)
df_convertidos = df_merged_dias.dropna(subset=['dias_a_primera_txn'])
df_convertidos

In [ ]:

# Total de usuarios que convirtieron
total_convertidos = len(df_convertidos)
total_convertidos

In [ ]:

# Conversiones en 1 / 7 / 30 días
pct_1_dia = (df_convertidos['dias_a_primera_txn'] <= 1).sum() / total_convertidos * 100
pct_7_dias = (df_convertidos['dias_a_primera_txn'] <= 7).sum() / total_convertidos * 100
pct_30_dias = (df_convertidos['dias_a_primera_txn'] <= 30).sum() / total_convertidos * 100

display(pct_1_dia, pct_7_dias, pct_30_dias)

| Métrica     | % Conversión | Interpretación                                                                                                                                                                           |
| ----------- | ------------ | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **1 día**   | 15.3%        | Una parte razonable convierte muy pronto, lo que indica que el proceso de onboarding o interés inicial funciona para algunos usuarios.                                                   |
| **7 días**  | 20.7%        | Solo 5.4% adicional convierte entre el día 2 y 7. Podría sugerir fricción temprana (por ejemplo: falta de valor claro o problemas de activación).                                        |
| **30 días** | 40.7%        | Un salto de \~20% más entre el día 8 y 30 sugiere que algunos usuarios toman más tiempo en decidir, quizá esperando una promoción, necesidad puntual, o explorando más antes de confiar. |

# Evolución semanal de usuarios nuevos vs. activos


In [ ]:
# Agrupar usuarios nuevos por semana

# Asegurar que la fecha está en formato datetime
df['create_date_user'] = pd.to_datetime(df['create_date_user'])

# Crear columna de semana de creación
df['semana_creacion_usuario'] = df['create_date_user'].dt.to_period('W').apply(lambda r: r.start_time)

# Agrupar por semana
usuarios_nuevos = df.groupby('semana_creacion_usuario')['user_id'].nunique().reset_index()
usuarios_nuevos.columns = ['semana', 'usuarios_nuevos']

display(df, usuarios_nuevos)


In [ ]:
# Agrupar usuarios activos por semana

df['create_date_transaction'] = pd.to_datetime(df['create_date_transaction'])

# Eliminar filas con NaT en la transacción antes de crear semana
df_filtrado = df.dropna(subset=['create_date_transaction']).copy()
df_filtrado['semana_transaccion'] = df_filtrado['create_date_transaction'].dt.to_period('W').apply(lambda r: r.start_time)

# Agrupar por semana, usuarios únicos que transaccionaron
usuarios_activos = df_filtrado.groupby('semana_transaccion')['user_id'].nunique().reset_index()
usuarios_activos.columns = ['semana', 'usuarios_activos']



In [ ]:
df_usuarios = pd.merge(usuarios_nuevos, usuarios_activos, on='semana', how='outer')
df_usuarios = df_usuarios.sort_values('semana').fillna(0)

df_usuarios


In [ ]:
df_usuarios_largo = df_usuarios.melt(id_vars='semana', value_vars=['usuarios_nuevos', 'usuarios_activos'],
                                     var_name='tipo_usuario', value_name='cantidad')

fig = px.line(df_usuarios_largo,
              x='semana',
              y='cantidad',
              color='tipo_usuario',
              markers=True,
              title='Evolución semanal de usuarios nuevos vs. activos')

fig.update_layout(xaxis_title='Semana',
                  yaxis_title='Cantidad de usuarios',
                  legend_title='Tipo de usuario')

fig.show()

# st.plotly_chart(fig, use_container_width=True)


🧠 Insights

Tendencias de crecimiento
Puedes ver si estás ganando más usuarios nuevos semana a semana (línea azul).

Engagement o retención
La línea roja muestra si los usuarios siguen usando el producto. Si esta crece más rápido que la azul, es señal de buena retención o reactivación.

Momentos críticos o picos
Por ejemplo, alrededor de marzo y noviembre 2018, hay picos en usuarios nuevos. Eso podría coincidir con campañas o lanzamientos.

Caídas abruptas
El desplome en mayo 2019 en ambos casos podría ser por datos incompletos en la última semana del dataset.

In [ ]:
print(df['create_date_transaction'].max())
print(df['create_date_user'].max())

In [ ]:
df_ultimas_fechas = df[df['create_date_transaction'] >= '2019-05-01']
print(df_ultimas_fechas['create_date_transaction'].value_counts())


In [ ]:
df['dia_create_transaction'] = df['create_date_transaction'].dt.date
df.groupby('dia_create_transaction')['user_id'].count().plot(kind='bar', figsize=(15,4))


In [ ]:
from datetime import datetime

# Convertir a fecha
df['create_date_transaction'] = pd.to_datetime(df['create_date_transaction'])

# Filtrar semana del 6 al 12 de mayo
mask = (df['create_date_transaction'] >= '2019-05-06') & (df['create_date_transaction'] <= '2019-05-12')
df_semana_caida = df[mask]
print(df_semana_caida[['user_id', 'create_date_transaction']].head(10))
print(df_semana_caida['user_id'].nunique())


🧠

Esto puede significar una de las siguientes cosas:

El sistema fue desconectado o dejó de registrar eventos 7 de enero de 2019

Hubo una migración, limpieza o interrupción de datos.

Ambiente de prueba: hay usuarios duplicados con muchos eventos el mismo día/hora, lo cual no es común en producción.

Tu dataset termina en esa fecha, y no es un problema de usuarios sino de cobertura temporal.

In [ ]:
df.columns

In [ ]:
# df.to_csv('../data/df_n.csv')

# 3. 👥 Perfiles de usuario y uso del producto

plan

In [ ]:
# Eliminar duplicados de usuarios
df_unicos = df.drop_duplicates(subset='user_id')

# Filtrar solo usuarios que tienen plan definido (no nulo ni vacío)
df_planes = df_unicos[df_unicos['plan'].notna() & (df_unicos['plan'] != '')]

# Contar usuarios únicos por tipo de plan
distribucion_planes = df_planes['plan'].value_counts().reset_index()
distribucion_planes.columns = ['plan', 'usuarios']

# Gráfica de barras
fig = px.bar(
    distribucion_planes,
    x='plan',
    y='usuarios',
    title='Distribución de usuarios por plan',
    color='plan',
    text='usuarios',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='Plan', yaxis_title='Número de usuarios')


edad

In [ ]:
df_edad = df[df['age_group'].notna()]

# Agrupar por grupo de edad y contar usuarios únicos
edad_grouped = (
    df_edad.groupby('age_group')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values(by='usuarios', ascending=False)
)

fig = px.bar(
    edad_grouped,
    x='age_group',
    y='usuarios',
    color='age_group',
    title='Usuarios únicos por grupo de edad',
    labels={'age_group': 'Grupo de edad', 'usuarios': 'Usuarios únicos'},
    color_discrete_sequence=px.colors.sequential.Sunset
)

fig.show()

__Por país__

In [ ]:
df['country']

In [ ]:
# Filtrar nulos
df_pais = df[df['country'].notna()]

# Agrupar por país y contar usuarios únicos
pais_grouped = (
    df_pais.groupby('country')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
)

pais_grouped

In [ ]:

fig = px.choropleth(
    pais_grouped,
    locations='country',  # Debe ser código ISO alpha-2 (como FR, MX)
    color='usuarios',
    hover_name='country',
    color_continuous_scale='Blues',
    labels={'usuarios': 'Usuarios únicos'},
    title='Distribución de usuarios por país'
)

fig.show()

In [ ]:
df_pais = pd.read_csv('https://datahub.io/core/country-list/r/data.csv')

df_pais

In [ ]:
lat_lon_por_pais = {
    'AT': {'lat': 47.5162, 'lon': 14.5501},   # Austria
    'AU': {'lat': -25.2744, 'lon': 133.7751}, # Australia
    'BE': {'lat': 50.5039, 'lon': 4.4699},    # Belgium
    'BG': {'lat': 42.7339, 'lon': 25.4858},   # Bulgaria
    'CH': {'lat': 46.8182, 'lon': 8.2275},    # Switzerland
    'CY': {'lat': 35.1264, 'lon': 33.4299},   # Cyprus
    'CZ': {'lat': 49.8175, 'lon': 15.4730},   # Czech Republic
    'DE': {'lat': 51.1657, 'lon': 10.4515},   # Germany
    'DK': {'lat': 56.2639, 'lon': 9.5018},    # Denmark
    'EE': {'lat': 58.5953, 'lon': 25.0136},   # Estonia
    'ES': {'lat': 40.4637, 'lon': -3.7492},   # Spain
    'FI': {'lat': 61.9241, 'lon': 25.7482},   # Finland
    'FR': {'lat': 46.6034, 'lon': 1.8883},    # France
    'GB': {'lat': 55.3781, 'lon': -3.4360},   # United Kingdom
    'GF': {'lat': 3.9339, 'lon': -53.1258},   # French Guiana
    'GG': {'lat': 49.4482, 'lon': -2.5895},   # Guernsey
    'GI': {'lat': 36.1408, 'lon': -5.3536},   # Gibraltar
    'GP': {'lat': 16.2650, 'lon': -61.5510},  # Guadeloupe
    'GR': {'lat': 39.0742, 'lon': 21.8243},   # Greece
    'HR': {'lat': 45.1000, 'lon': 15.2000},   # Croatia
    'HU': {'lat': 47.1625, 'lon': 19.5033},   # Hungary
    'IE': {'lat': 53.4129, 'lon': -8.2439},   # Ireland
    'IM': {'lat': 54.2361, 'lon': -4.5481},   # Isle of Man
    'IS': {'lat': 64.9631, 'lon': -19.0208},  # Iceland
    'IT': {'lat': 41.8719, 'lon': 12.5674},   # Italy
    'JE': {'lat': 49.2144, 'lon': -2.1313},   # Jersey
    'LI': {'lat': 47.1660, 'lon': 9.5554},    # Liechtenstein
    'LT': {'lat': 55.1694, 'lon': 23.8813},   # Lithuania
    'LU': {'lat': 49.8153, 'lon': 6.1296},    # Luxembourg
    'LV': {'lat': 56.8796, 'lon': 24.6032},   # Latvia
    'MQ': {'lat': 14.6415, 'lon': -61.0242},  # Martinique
    'MT': {'lat': 35.9375, 'lon': 14.3754},   # Malta
    'NL': {'lat': 52.1326, 'lon': 5.2913},    # Netherlands
    'NO': {'lat': 60.4720, 'lon': 8.4689},    # Norway
    'PL': {'lat': 51.9194, 'lon': 19.1451},   # Poland
    'PT': {'lat': 39.3999, 'lon': -8.2245},   # Portugal
    'RE': {'lat': -21.1151, 'lon': 55.5364},  # Réunion
    'RO': {'lat': 45.9432, 'lon': 24.9668},   # Romania
    'SE': {'lat': 60.1282, 'lon': 18.6435},   # Sweden
    'SI': {'lat': 46.1512, 'lon': 14.9955},   # Slovenia
    'SK': {'lat': 48.6690, 'lon': 19.6990},   # Slovakia
}


In [ ]:
# Agrupar usuarios por código de país
df_pais = df[df['country'].notna()]
usuarios_por_pais = (
    df_pais.groupby('country')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
)

# Agregar columnas de latitud y longitud
usuarios_por_pais['lat'] = usuarios_por_pais['country'].map(lambda x: lat_lon_por_pais.get(x, {}).get('lat'))
usuarios_por_pais['lon'] = usuarios_por_pais['country'].map(lambda x: lat_lon_por_pais.get(x, {}).get('lon'))

# Quitar filas con coordenadas faltantes
usuarios_por_pais = usuarios_por_pais.dropna(subset=['lat', 'lon'])

usuarios_por_pais


In [ ]:
fig = px.scatter_geo(
    usuarios_por_pais,
    lat='lat',
    lon='lon',
    size='usuarios',
    hover_name='country',
    projection='natural earth',
    color='usuarios',
    color_continuous_scale='Viridis',
    title='Distribución de usuarios por país (burbuja proporcional)',
    scope='europe',
    size_max=70  # 👈 aumenta el tamaño máximo de burbuja
)

fig.update_layout(
    width=1000,
    height=600,
    margin={"r":0,"t":40,"l":0,"b":0}
)

fig.show()


In [ ]:
df = df.merge(
    usuarios_por_pais[['country', 'lat', 'lon']],  # solo traemos lo necesario
    on='country',
    how='left'
)

# df.to_csv('../data/df_n.csv')


In [ ]:
df

__por ciudad__

In [ ]:
# Agrupar por ciudad y contar usuarios únicos
usuarios_por_ciudad = (
    df.groupby('city')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values('usuarios', ascending=False)
)

usuarios_por_ciudad


In [ ]:
# Filtrar solo las ciudades con más usuarios
top_ciudades = usuarios_por_ciudad.head(20)
top_ciudades

In [ ]:
# visualización
fig = px.bar(
    top_ciudades,
    x='usuarios',
    y='city',
    orientation='h',
    color='usuarios',
    color_continuous_scale='Tealgrn',
    title='Top 20 ciudades con más usuarios únicos',
    labels={'city': 'Ciudad', 'usuarios': 'Usuarios únicos'}
)
fig.show()

__Distribución por canal__

In [ ]:
# Agrupar por canal y contar usuarios únicos
usuarios_por_canal = (
    df.groupby('channel')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values('usuarios', ascending=False)
)
usuarios_por_canal

In [ ]:
# Visualizar
fig = px.bar(
    usuarios_por_canal,
    x='channel',
    y='usuarios',
    color='usuarios',
    title='Distribución de usuarios por canal',
    labels={'channel': 'Canal de adquisición', 'usuarios': 'Usuarios únicos'},
    color_continuous_scale='burg'
)
fig.show()

__por device__

In [ ]:
#Agrupar por dispositivo y contar usuarios únicos
usuarios_por_dispositivo = (
    df.groupby('brand_device')['user_id']
    .nunique()
    .reset_index()
    .rename(columns={'user_id': 'usuarios'})
    .sort_values(by='usuarios', ascending=False)
)
usuarios_por_dispositivo

In [ ]:
# visualización
fig = px.bar(
    usuarios_por_dispositivo,
    x='brand_device',
    y='usuarios',
    title='Distribución de usuarios por tipo de dispositivo',
    labels={'usuarios': 'Usuarios únicos', 'brand_device': 'Dispositivo'},
    color='usuarios',
    color_continuous_scale='cividis'
)
fig.show()



# Número de transacciones por segmento

In [ ]:
# Agrupar por usuario para obtener una fila por user_id con su edad y total de transacciones
usuarios_unicos = df.groupby('user_id').agg({
    'age_group': 'first',
    'num_transactions': 'first'  # o 'sum', dependiendo cómo esté el dato original
}).reset_index()

# Ahora agrupar por grupo de edad
transacciones_por_edad = usuarios_unicos.groupby('age_group')['num_transactions'].sum().reset_index()

# nostrar
df_transactions = pd.read_csv('../data/transactions.csv')
df_txn = df_transactions['transaction_id'].nunique()
display(usuarios_unicos, transacciones_por_edad, transacciones_por_edad['num_transactions'].sum(), df_txn)

In [ ]:
# Crear gráfico de barras
fig = px.bar(
    transacciones_por_edad,
    x='age_group',
    y='num_transactions',
    title='Número total de transacciones por grupo de edad',
    labels={'age_group': 'Grupo de edad', 'num_transactions': 'Total de transacciones'},
    color='num_transactions',
    color_continuous_scale='viridis'
)
fig.show()

# % de conversión por grupo

In [ ]:
usuarios_unicos = df.drop_duplicates(subset='user_id')
usuarios_unicos

In [ ]:
# Agrupar por plan y calcular métricas de conversión
conversion_por_plan = usuarios_unicos.groupby('plan')['converted'].agg(['count', 'sum']).reset_index()
conversion_por_plan['conversion_rate'] = (conversion_por_plan['sum'] / conversion_por_plan['count']) * 100
conversion_por_plan

In [ ]:
# visualizar
fig = px.bar(
    conversion_por_plan,
    x='plan',
    y='conversion_rate',
    title='% de conversión por tipo de plan',
    labels={'conversion_rate': '% de conversión', 'plan': 'Tipo de plan'},
    text='conversion_rate',
    color='conversion_rate',
    color_continuous_scale='gnbu'
)
fig.show()

METAL tiene una tasa de conversión de ~1.18%, lo que significa que 1.18 de cada 100 usuarios con plan METAL convirtieron.

STANDARD tiene una tasa de conversión más baja: ~0.66%.

PREMIUM_FREE y PREMIUM_OFFER están en 0%, posiblemente porque nadie con esos planes convirtió o hay muy pocos usuarios.

In [ ]:
df.columns

# Uso de funcionalidades crypto

In [ ]:
# Agrupar por uso de cripto y contar usuarios únicos
uso_crypto = df.groupby('user_settings_crypto_unlocked')['user_id'].nunique().reset_index()
uso_crypto.columns = ['Crypto habilitado', 'Usuarios únicos']

uso_crypto

In [ ]:
# Convertir valores a etiquetas legibles
uso_crypto['Crypto habilitado'] = uso_crypto['Crypto habilitado'].map({
    'True': 'Sí',
    'False': 'No',
    True: 'Sí',
    False: 'No'
})

uso_crypto


In [ ]:
# Gráfico de pastel
fig = px.pie(
    uso_crypto,
    values='Usuarios únicos',
    names='Crypto habilitado',
    title='Distribución de usuarios con funcionalidad cripto activada',
    color_discrete_sequence=px.colors.sequential.RdBu
)

fig.show()

In [ ]:
print(df[['country', 'plan', 'age']].info())
print(df[['country', 'plan']].dropna().nunique())

In [ ]:
print(repr(df['country'].unique()))  # <- esto revela si hay espacios ocultos

In [ ]:
# Supón que eliges uno de los países válidos
pais_seleccionado = 'FR'  # o cualquier otro que veas en df['country'].unique()

# Supón que eliges algunos planes
planes_seleccionados = ['STANDARD', 'PREMIUM', 'METAL']  # ajusta según tu dataset

# Rango de edad (simulado)
edad_min = int(df['age'].min())
edad_max = int(df['age'].max())
rango_edad = (edad_min, edad_max)  # o (24, 65), etc.

print("Total:", len(df))

df1 = df[df['country'] == pais_seleccionado]
print("Filtrado por país:", len(df1))

df2 = df1[df1['plan'].isin(planes_seleccionados)]
print("Filtrado por plan:", len(df2))

df3 = df2[df2['age'].between(rango_edad[0], rango_edad[1])]
print("Filtrado por edad:", len(df3))

df_filtrado = df3
df_filtrado.head()  # Mostrar muestra


# 4. ⚠️ Identificación de churn y recomendaciones

__churn__

In [ ]:
# Paso 1: Última fecha del dataset
fecha_max = df['create_date_transaction'].max()

fecha_max

In [ ]:

# Paso 2: Fecha de última transacción por usuario
ultima_txn = df.groupby('user_id')['create_date_transaction'].max().reset_index()
ultima_txn.columns = ['user_id', 'ultima_txn']
ultima_txn


In [ ]:

# Paso 3: Calcular días de inactividad
ultima_txn['dias_inactivo'] = (fecha_max - ultima_txn['ultima_txn']).dt.days
ultima_txn

In [ ]:
# Paso 4: Marcar usuarios como churned si llevan más de 60 días sin transaccionar
ultima_txn['churned'] = ultima_txn['dias_inactivo'] > 90
ultima_txn

In [ ]:
# Paso 5: Churn rate
churn_rate = ultima_txn['churned'].mean() * 100
churn_rate

# calcular churn en df

In [ ]:
# Paso 1: Obtener la última fecha de transacción del dataset
fecha_corte = df['create_date_transaction'].max()
fecha_corte

In [ ]:

# Paso 2: Convertir fechas a naive (sin timezone), por si acaso
fecha_corte = fecha_corte.tz_localize(None) if fecha_corte.tzinfo else fecha_corte
df['create_date_transaction'] = df['create_date_transaction'].dt.tz_localize(None)

In [ ]:

# Paso 3: Calcular días desde última transacción
df['dias_inactivo'] = (fecha_corte - df['create_date_transaction']).dt.days
df

In [ ]:

# Paso 4: Marcar como churned si lleva más de 60 días sin transaccionar
df['churned'] = df['dias_inactivo'] > 60

In [ ]:
df[(df['has_transaction'] == True) & (df['churned'] == True)]

In [ ]:

# df.to_csv('../data/df_n.csv')

In [ ]:
# Filtrar usuarios notificados (por push o email) pero que NO se convirtieron
notificados_no_convertidos = df[
    ((df['attributes_notifications_marketing_push'] == 1) | 
     (df['attributes_notifications_marketing_email'] == 1)) & 
    (df['converted'] == 0)
]
notificados_no_convertidos


In [ ]:

# Crear nueva columna con canal de notificación
def canal(row):
    if row['attributes_notifications_marketing_push'] == 1 and row['attributes_notifications_marketing_email'] == 1:
        return 'Ambos'
    elif row['attributes_notifications_marketing_push'] == 1:
        return 'Push'
    elif row['attributes_notifications_marketing_email'] == 1:
        return 'Email'
    else:
        return 'Ninguno'

notificados_no_convertidos['canal_notificado'] = notificados_no_convertidos.apply(canal, axis=1)

notificados_no_convertidos

In [ ]:
# Agrupar por canal
canal_counts = notificados_no_convertidos['canal_notificado'].value_counts().reset_index()
canal_counts.columns = ['Canal', 'Usuarios no convertidos']

canal_counts

In [ ]:

# Graficar
fig = px.pie(
    canal_counts, 
    names='Canal', 
    values='Usuarios no convertidos', 
    title='Usuarios notificados pero no convertidos (por canal)'
)
fig.show()

In [ ]:
df

# Usuarios notificados pero no convertidos por tipo de plan

In [ ]:
# Filtrar usuarios notificados pero no convertidos
notificados_no_convertidos = df[(df['has_notification'] == True) & (df['converted'] == 0)]
notificados_no_convertidos

In [ ]:

# Agrupar por plan
usuarios_por_plan = notificados_no_convertidos.groupby('plan')['user_id'].nunique().reset_index()
usuarios_por_plan.columns = ['plan', 'usuarios']

usuarios_por_canal


In [ ]:

# Visualización
fig = px.bar(
    usuarios_por_plan,
    x='plan',
    y='usuarios',
    color='plan',
    title='Usuarios notificados pero no convertidos por tipo de plan',
    labels={'usuarios': 'Usuarios únicos', 'plan': 'Tipo de plan'}
)

fig.show()

# Usuarios notificados pero no convertidos por grupo de edad

In [ ]:
# Filtrar usuarios notificados pero no convertidos
notificados_no_convertidos = df[(df['has_notification'] == True) & (df['converted'] == 0)]
notificados_no_convertidos

In [ ]:

# Agrupar por grupo de edad
usuarios_por_edad = notificados_no_convertidos.groupby('age_group')['user_id'].nunique().reset_index()
usuarios_por_edad.columns = ['age_group', 'usuarios']
usuarios_por_edad

In [ ]:

# Ordenar grupos de edad si son strings
usuarios_por_edad = usuarios_por_edad.sort_values(by='age_group')
usuarios_por_edad

In [ ]:

# Visualizar
fig = px.bar(
    usuarios_por_edad,
    x='age_group',
    y='usuarios',
    color='age_group',
    title='Usuarios notificados pero no convertidos por grupo de edad',
    labels={'usuarios': 'Usuarios únicos', 'age_group': 'Grupo de edad'}
)

fig.show()

# Usuarios completamente inactivos (sin ninguna transacción) por grupo de edad

In [ ]:
import plotly.express as px

# Filtrar usuarios sin transacciones
usuarios_inactivos = df[df['has_transaction'] == False]
usuarios_inactivos


In [ ]:

# Agrupar por grupo de edad (puedes cambiarlo a 'plan', 'channel', etc.)
inactivos_por_edad = usuarios_inactivos.groupby('age_group')['user_id'].nunique().reset_index()
inactivos_por_edad.columns = ['age_group', 'usuarios']
inactivos_por_edad

In [ ]:

# Ordenar si es necesario
inactivos_por_edad = inactivos_por_edad.sort_values(by='age_group')
inactivos_por_edad

In [ ]:

# Visualizar
fig = px.bar(
    inactivos_por_edad,
    x='age_group',
    y='usuarios',
    color='age_group',
    title='Usuarios completamente inactivos (sin ninguna transacción) por grupo de edad',
    labels={'usuarios': 'Usuarios únicos', 'age_group': 'Grupo de edad'}
)

fig.show()

# inactivos (sin ninguna transacción) por plan

In [ ]:
usuarios_inactivos = df[df['has_transaction'] == False]
usuarios_inactivos

In [ ]:
inactivos_por_plan = usuarios_inactivos.groupby('plan')['user_id'].nunique().reset_index()
inactivos_por_plan.columns = ['plan', 'usuarios_inactivos']
inactivos_por_plan

In [ ]:
fig = px.bar(
    inactivos_por_plan.sort_values(by='usuarios_inactivos', ascending=False),
    x='plan',
    y='usuarios_inactivos',
    title='Usuarios inactivos por tipo de plan',
    labels={'usuarios_inactivos': 'Usuarios inactivos', 'plan': 'Tipo de plan'},
    color='usuarios_inactivos',
    color_continuous_scale='viridis'
)
fig.show()


# inactivos (que nunca hicieron transacción) agrupados por canal

In [ ]:
usuarios_inactivos = df[df['has_transaction'] == False]
usuarios_inactivos

In [ ]:
inactivos_por_canal = usuarios_inactivos.groupby('channel')['user_id'].nunique().reset_index()
inactivos_por_canal.columns = ['channel', 'usuarios_inactivos']
inactivos_por_canal

In [ ]:
fig = px.bar(
    inactivos_por_canal.sort_values(by='usuarios_inactivos', ascending=False),
    x='channel',
    y='usuarios_inactivos',
    title='Usuarios inactivos por canal de adquisición',
    labels={'usuarios_inactivos': 'Usuarios inactivos', 'channel': 'Canal'},
    color='usuarios_inactivos',
    color_continuous_scale='plasma'
)
fig.show()

# Días sin actividad desde alta

In [ ]:
usuarios_sin_txn = df[df['has_transaction'] == False].copy()
usuarios_sin_txn

In [ ]:
fecha_max = df['create_date_transaction'].max()
fecha_max


In [ ]:
# Obtener la zona horaria desde create_date_user
tz = usuarios_sin_txn['create_date_user'].dt.tz
tz


In [ ]:

# Asegurar que la fecha máxima tenga la misma zona horaria
fecha_max = pd.Timestamp(df['create_date_transaction'].max(), tz=tz)
fecha_max

In [ ]:

# Calcular días sin actividad
usuarios_sin_txn['dias_sin_actividad'] = (fecha_max - usuarios_sin_txn['create_date_user']).dt.days
usuarios_sin_txn

In [ ]:
bins = [0, 7, 14, 30, 60, 90, 180, 365, 9999]
labels = ['0-7 días', '8-14 días', '15-30 días', '31-60 días', '61-90 días', '91-180 días', '181-365 días', '365+ días']

usuarios_sin_txn['rango_dias_sin_actividad'] = pd.cut(usuarios_sin_txn['dias_sin_actividad'], bins=bins, labels=labels)
dias_inactivos = usuarios_sin_txn.groupby('rango_dias_sin_actividad')['user_id'].nunique().reset_index()
dias_inactivos.columns = ['rango_dias', 'usuarios']


In [ ]:
fig = px.bar(
    dias_inactivos,
    x='rango_dias',
    y='usuarios',
    title='Usuarios sin actividad por rango de días desde su alta',
    labels={'rango_dias': 'Días sin actividad', 'usuarios': 'Usuarios únicos'},
    color='usuarios',
    color_continuous_scale='magma'
)
fig.show()


In [ ]:
df.groupby('churned')['user_id'].nunique()

In [ ]:
df

In [ ]:
df_unicos = df.drop_duplicates(subset='user_id', keep='first')
df_con_transaccion = df_unicos[df_unicos['has_transaction'] == True]
df_con_transaccion.groupby('churned')['user_id'].nunique()

# dashboard v2.0

In [ ]:
# Asume que df es tu DataFrame con usuarios únicos y las columnas calculadas

# Métrica 1: Usuarios únicos
total_users = df['user_id'].nunique()
total_users

In [ ]:

# Métrica 2: % de conversión (usuarios que convirtieron al menos una vez)
conversion_rate = (df['converted'].sum() / total_users) * 100
conversion_rate

In [ ]:

# Métrica 3: % de churn (usuarios activos que luego abandonaron)
# Asegúrate de usar un df de usuarios únicos
df_unicos = df.drop_duplicates(subset='user_id')

total_users = df_unicos['user_id'].nunique()
churned_users = df_unicos[(df_unicos['churned'] == True) & (df_unicos['has_transaction'] == True)]['user_id'].nunique()

churn_rate = (churned_users / total_users) * 100

display(df_unicos, total_users, churned_users, churn_rate)


In [ ]:
# Métrica 4: Promedio de días a la primera transacción (entre usuarios que transaccionaron)

# 1) Obtener la primera transacción de cada usuario
df_txn = df.dropna(subset=['create_date_transaction'])
df_first_txn = df_txn.groupby('user_id')['create_date_transaction'].min().reset_index()
df_first_txn.columns = ['user_id', 'first_transaction_date']

# 2) Traer fecha de creación de usuario
df_user_creation = df[['user_id', 'create_date_user']].drop_duplicates()

# 3) Merge para calcular días a la primera transacción
df_dias_txn = pd.merge(df_user_creation, df_first_txn, on='user_id', how='left')

# --- ⚠️ Corregir zona horaria para evitar errores ---
df_dias_txn['first_transaction_date'] = df_dias_txn['first_transaction_date'].dt.tz_localize(None)
df_dias_txn['create_date_user'] = df_dias_txn['create_date_user'].dt.tz_localize(None)

# Calcular días a la primera transacción
df_dias_txn['dias_a_primera_txn'] = (
    df_dias_txn['first_transaction_date'] - df_dias_txn['create_date_user']
).dt.days

# 4) Merge con el df principal para incorporar dias_a_primera_txn
df_copy = df.drop_duplicates(subset='user_id').merge(
    df_dias_txn[['user_id', 'dias_a_primera_txn']],
    on='user_id',
    how='left'
)

avg_days_to_first_txn = df_copy['dias_a_primera_txn'].dropna().mean()

avg_days_to_first_txn


In [ ]:
df

# Evolución semanal de usuarios nuevos vs. activos

In [ ]:
# 1) Asegúrate de que las fechas de transacción están en datetime
df_n_i = df.copy()
df_n_i['create_date_transaction'] = pd.to_datetime(df_n_i['create_date_transaction'], errors='coerce')



In [ ]:

# 2) Crea la columna de semana de la transacción (usando inicio de semana: lunes)
df_n_i['semana_transaccion'] = df_n_i['create_date_transaction'].dt.to_period('W').apply(
    lambda r: r.start_time if pd.notnull(r) else pd.NaT
)


In [ ]:

# 3) Usuarios activos por semana (usuarios únicos que transaccionaron)
usuarios_activos = (
    df_n_i.dropna(subset=['semana_transaccion'])  # Asegura eliminar NaT aquí
      .groupby('semana_transaccion')['user_id']
      .nunique()
      .reset_index(name='usuarios_activos')
)
usuarios_activos

In [ ]:

# 4) Usuarios nuevos por semana (basado en fecha de creación de usuario)
df_n_i['create_date_user'] = pd.to_datetime(df_n_i['create_date_user'], errors='coerce')
df_n_i['semana_creacion'] = df_n_i['create_date_user'].dt.to_period('W').apply(
    lambda r: r.start_time if pd.notnull(r) else pd.NaT
)
usuarios_nuevos = (
    df_n_i.dropna(subset=['semana_creacion'])  # Asegura eliminar NaT aquí también
      .groupby('semana_creacion')['user_id']
      .nunique()
      .reset_index(name='usuarios_nuevos')
)

usuarios_nuevos

In [ ]:

# 5) Fusiona ambos resultados
df_evolucion = pd.merge(
    usuarios_nuevos,
    usuarios_activos,
    left_on='semana_creacion',
    right_on='semana_transaccion',
    how='outer'
)

df_evolucion

In [ ]:

# 6) Unifica la columna de semana
df_evolucion['semana'] = df_evolucion['semana_creacion'].combine_first(df_evolucion['semana_transaccion'])


In [ ]:

# 7) Elimina semanas NaT o inválidas
df_evolucion = df_evolucion.dropna(subset=['semana']).sort_values('semana')


In [ ]:

# 8) Graficar con Plotly Express
fig = px.line(
    df_evolucion,
    x='semana',
    y=['usuarios_nuevos', 'usuarios_activos'],
    labels={'value': 'Cantidad de usuarios', 'variable': 'Tipo de usuario', 'semana': 'Semana'},
    title='Evolución semanal de usuarios nuevos vs. activos'
)
fig.update_layout(xaxis=dict(tickformat='%Y-%m-%d'))  # Formato de fechas en eje X
fig.show()

In [ ]:
print("Última fecha de creación de usuario:", df['create_date_user'].max())
print("Última fecha de transacción:", df['create_date_transaction'].max())

In [ ]:
df_evolucion

In [ ]:
df_evolucion['usuarios_nuevos'].sum(), df_evolucion['usuarios_activos'].sum()

In [ ]:
df_evolucion['usuarios_nuevos'].sum(), df_evolucion['usuarios_activos'].sum()

In [ ]:
df

In [ ]:
# 1) Cohorte: semana de registro
df['cohort_week'] = df['create_date_user'].dt.to_period('W').apply(lambda r: r.start_time)
df


In [ ]:

# 2) Semana de actividad (semana de transacción)
df['activity_week'] = df['create_date_transaction'].dt.to_period('W').apply(lambda r: r.start_time if pd.notnull(r) else pd.NaT)
df

In [ ]:

# 3) Filtrar solo usuarios con transacción
df_active = df.dropna(subset=['activity_week'])
df_active

In [ ]:

# 4) Calcular semanas desde el registro correctamente
df_active['weeks_since_signup'] = (
    (df_active['activity_week'] - df_active['cohort_week']).dt.days // 7
)
df_active

In [ ]:

# 5) Usuarios únicos por cohorte y semana desde registro
retention = (
    df_active.groupby(['cohort_week', 'weeks_since_signup'])['user_id']
    .nunique()
    .reset_index(name='active_users')
)
retention

In [ ]:

# 6) Número de usuarios en cada cohorte (tamaño base)
cohort_sizes = (
    df.groupby('cohort_week')['user_id']
    .nunique()
    .reset_index(name='cohort_size')
)
cohort_sizes

In [ ]:

# 7) Merge para calcular % de retención
retention = retention.merge(cohort_sizes, on='cohort_week')
retention['retention_rate'] = retention['active_users'] / retention['cohort_size']

retention

In [ ]:

# 8) Pivotear para matriz de retención
retention_matrix = retention.pivot(index='cohort_week', columns='weeks_since_signup', values='retention_rate')
retention_matrix

In [ ]:

fig = px.imshow(
    retention_matrix,
    labels=dict(x='Semanas desde registro', y='Cohorte de registro', color='Tasa de retención'),
    color_continuous_scale='Blues',
    title='Curva de retención por cohortes semanales'
)
fig.show()

# Primera transacción

In [ ]:
# 1) Calcular días hasta primera transacción
df_merged = df.dropna(subset=['create_date_transaction']).copy()
df_merged['create_date_transaction'] = df_merged['create_date_transaction'].dt.tz_localize(None)
df_merged['create_date_user'] = df_merged['create_date_user'].dt.tz_localize(None)
df_merged['dias_a_primera_txn'] = (df_merged['create_date_transaction'] - df_merged['create_date_user']).dt.days

df_merged

In [ ]:

# Ahora calcula los días a la primera transacción
df_merged['dias_a_primera_txn'] = (
    df_merged['create_date_transaction'] - df_merged['create_date_user']
).dt.days

df_merged

In [ ]:

# 2) Quedarse solo con el primer registro de cada usuario para no duplicar conversiones
df_first_txn = df_merged.sort_values('dias_a_primera_txn').drop_duplicates(subset='user_id', keep='first')
df_first_txn

In [ ]:

# 3) Crear columnas booleanas: si convirtió en 1/7/30 días
df_first_txn['converted_1d'] = df_first_txn['dias_a_primera_txn'] <= 1
df_first_txn['converted_7d'] = df_first_txn['dias_a_primera_txn'] <= 7
df_first_txn['converted_30d'] = df_first_txn['dias_a_primera_txn'] <= 30
df_first_txn['converted_60d'] = df_first_txn['dias_a_primera_txn'] <= 60
df_first_txn['converted_90d'] = df_first_txn['dias_a_primera_txn'] <= 90
df_first_txn['converted_120d'] = df_first_txn['dias_a_primera_txn'] <= 120
df_first_txn['converted_mayor_120d'] = df_first_txn['dias_a_primera_txn'] > 120
df_first_txn

In [ ]:

# 4) Calcular el total de usuarios registrados
total_users = df['user_id'].nunique()
total_users

In [ ]:

# 5) Calcular tasas de conversión correctamente
conversion_1d = df_first_txn['converted_1d'].sum() / total_users * 100
conversion_7d = df_first_txn['converted_7d'].sum() / total_users * 100
conversion_30d = df_first_txn['converted_30d'].sum() / total_users * 100
conversion_60d = df_first_txn['converted_60d'].sum() / total_users * 100
conversion_90d = df_first_txn['converted_90d'].sum() / total_users * 100
conversion_120d = df_first_txn['converted_120d'].sum() / total_users * 100
converted_mayor_120d = df_first_txn['converted_mayor_120d'].sum() / total_users * 100

conversion_1d, conversion_7d, conversion_30d, conversion_60d, conversion_90d, conversion_120d, converted_mayor_120d

In [ ]:

conversion_data = {
    'Periodo': ['1 día', '7 días', '30 días', '60 días', '90 días', '120 días', 'Mayor a 120 días'],
    'Conversión (%)': [conversion_1d, conversion_7d, conversion_30d,  conversion_60d,  conversion_90d, conversion_120d, converted_mayor_120d]
}

conversion_data

In [ ]:

fig = px.bar(
    conversion_data,
    x='Periodo',
    y='Conversión (%)',
    text='Conversión (%)',
    title='% de usuarios que convierten en 1, 7 y 30 días',
    color='Periodo',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(yaxis_range=[0, 100])
fig.show()


Cerca del 35% de los usuarios convierten dentro de los primeros 120 días, mientras que un 65% nunca realiza una transacción o la hace mucho más tarde, lo que sugiere oportunidades en onboarding y activación temprana.

In [ ]:
# Filtrar usuarios con al menos una transacción
df_txn = df.dropna(subset=['create_date_transaction'])

# Obtener primera transacción por usuario
df_first_txn = (
    df_txn.groupby('user_id')['create_date_transaction']
    .min()
    .reset_index()
    .rename(columns={'create_date_transaction': 'first_transaction_date'})
)

df_txn, df_first_txn

In [ ]:
df_user_creation = df[['user_id', 'create_date_user']].drop_duplicates()
df_user_creation

In [ ]:
df_merged = df_user_creation.merge(
    df_first_txn,
    on='user_id',
    how='inner'  # Solo usuarios que tienen al menos una transacción
)
df_merged

In [ ]:
df_merged['create_date_user'] = df_merged['create_date_user'].dt.tz_localize(None)
df_merged['first_transaction_date'] = df_merged['first_transaction_date'].dt.tz_localize(None)


In [ ]:
df_merged['dias_a_primera_txn'] = (
    df_merged['first_transaction_date'] - df_merged['create_date_user']
).dt.days


In [ ]:
df_merged

In [ ]:
print(df_merged['dias_a_primera_txn'].describe())


In [ ]:
df_merged['user_id'].nunique()

In [ ]:
fig = px.histogram(
    df_merged,
    x='dias_a_primera_txn',
    nbins=300,
    title='Distribución del tiempo hasta la primera transacción',
    labels={'dias_a_primera_txn': 'Días hasta primera transacción'},
    color_discrete_sequence=['#00BFC4']
)

fig.update_layout(
    xaxis_title='Días hasta la primera transacción',
    yaxis_title='Número de usuarios',
    bargap=0.1
)

fig.show()

# Usuarios notificados pero no convertidos 

In [ ]:
df.columns

In [ ]:
# Filtrar usuarios notificados pero sin transacción
usuarios_no_convertidos = df[
    (df['has_notification'] == True) & (df['has_transaction'] == False)
].copy()
usuarios_no_convertidos

# notificados no convertidos

In [ ]:
df.groupby('has_notification')['user_id'].nunique()

In [ ]:
# Agrupar por canal y contar usuarios únicos
no_conv_por_canal = (
    usuarios_no_convertidos.groupby('channel')['user_id']
    .nunique()
    .reset_index(name='usuarios_no_convertidos')
    .sort_values('usuarios_no_convertidos', ascending=False)
)
no_conv_por_canal['usuarios_no_convertidos'].sum()

In [ ]:

# Graficar
fig = px.bar(
    no_conv_por_canal,
    x='channel',
    y='usuarios_no_convertidos',
    text='usuarios_no_convertidos',
    title='Usuarios notificados pero no convertidos por canal',
    labels={'channel': 'Canal', 'usuarios_no_convertidos': 'Usuarios sin conversión'}
)
fig.update_traces(textposition='outside')
fig.show()


In [ ]:
no_conv_por_plan = (
    usuarios_no_convertidos.groupby('plan')['user_id']
    .nunique()
    .reset_index(name='usuarios_no_convertidos')
    .sort_values('usuarios_no_convertidos', ascending=False)
)

fig = px.bar(
    no_conv_por_plan,
    x='plan',
    y='usuarios_no_convertidos',
    text='usuarios_no_convertidos',
    title='Usuarios notificados pero no convertidos por plan',
    labels={'plan': 'Plan', 'usuarios_no_convertidos': 'Usuarios sin conversión'}
)
fig.update_traces(textposition='outside')
fig.show()


In [ ]:
no_conv_por_edad = (
    usuarios_no_convertidos.groupby('age_group')['user_id']
    .nunique()
    .reset_index(name='usuarios_no_convertidos')
    .sort_values('age_group')
)

fig = px.bar(
    no_conv_por_edad,
    x='age_group',
    y='usuarios_no_convertidos',
    text='usuarios_no_convertidos',
    title='Usuarios notificados pero no convertidos por grupo de edad',
    labels={'age_group': 'Grupo de edad', 'usuarios_no_convertidos': 'Usuarios sin conversión'}
)
fig.update_traces(textposition='outside')
fig.show()


In [ ]:
# Agrupar por canal y estado de conversión
df_grouped = (
    df[df['has_notification']]
    .groupby(['channel', 'converted'])['user_id']
    .nunique()
    .reset_index()
    .pivot(index='channel', columns='converted', values='user_id')
    .fillna(0)
    .reset_index()
)
df_grouped


In [ ]:

df_grouped.columns = ['Canal', 'No_convirtieron', 'Convirtieron']  # Reordenar columnas
df_grouped

In [ ]:

fig = px.bar(
    df_grouped,
    x='Canal',
    y=['No_convirtieron', 'Convirtieron'],
    labels={'value': 'Usuarios', 'variable': 'Estado'},
    title='Usuarios notificados: Convirtieron vs. No Convirtieron por canal',
    barmode='stack',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.show()

In [ ]:
# Filtrar usuarios activos e inactivos
usuarios_activos = df[df['has_transaction'] == True]
usuarios_inactivos = df[df['has_transaction'] == False]

# Agrupar activos por grupo de edad
activos_por_edad = (
    usuarios_activos.groupby('age_group')['user_id']
    .nunique()
    .reset_index(name='usuarios_activos')
)

# Agrupar inactivos por grupo de edad
inactivos_por_edad = (
    usuarios_inactivos.groupby('age_group')['user_id']
    .nunique()
    .reset_index(name='usuarios_inactivos')
)

# Unir ambos dataframes
comparacion_edad = pd.merge(
    activos_por_edad, inactivos_por_edad,
    on='age_group', how='outer'
).fillna(0).sort_values('age_group')

# Graficar apilada
import plotly.express as px

fig_comparacion_edad = px.bar(
    comparacion_edad,
    x='age_group',
    y=['usuarios_activos', 'usuarios_inactivos'],
    barmode='stack',  # <-- ¡aquí el cambio importante!
    labels={
        'value': 'Usuarios',
        'variable': 'Estado',
        'age_group': 'Grupo de edad'
    },
    title='Usuarios activos vs. inactivos por grupo de edad (Apilado)',
    color_discrete_sequence=px.colors.qualitative.Set2_r,
    text_auto=True
)

fig_comparacion_edad.update_layout(yaxis_title='Número de usuarios')
fig_comparacion_edad.show()


In [ ]:
usuarios_activos['user_id'].nunique(), usuarios_inactivos['user_id'].nunique()

In [ ]:
comparacion_edad['usuarios_inactivos'].sum()

In [ ]:
# Agrupar usuarios únicos por estado de churn
churn_summary = (
    df.drop_duplicates(subset='user_id')
    .groupby('churned')['user_id']
    .nunique()
    .reset_index(name='usuarios')
)
churn_summary


In [ ]:

# Convertir True/False a etiquetas más amigables
churn_summary['Estado'] = churn_summary['churned'].map({True: 'Churned', False: 'No Churned'})
churn_summary

In [ ]:

# Crear gráfico de pastel
fig_churn = px.pie(
    churn_summary,
    names='Estado',
    values='usuarios',
    title='Distribución de usuarios churned vs. no churned',
    color='Estado',
    color_discrete_map={'Churned': '#FF6F61', 'No Churned': '#6BA292'},  # Colores personalizados
    hole=0.4  # Para hacer un donut chart
)

fig_churn.update_traces(textinfo='percent+label')

fig_churn.show()

In [ ]:
# Agrupar usuarios únicos por plan y estado de churn
churn_by_plan = (
    df.drop_duplicates(subset='user_id')
    .groupby(['plan', 'churned'])['user_id']
    .nunique()
    .reset_index(name='usuarios')
)
churn_by_plan


In [ ]:

# Convertir churn True/False a etiquetas legibles
churn_by_plan['Estado'] = churn_by_plan['churned'].map({True: 'Churned', False: 'No Churned'})
churn_by_plan['usuarios'].sum()

In [ ]:

# Graficar barras apiladas por plan
fig_churn_plan = px.bar(
    churn_by_plan,
    x='plan',
    y='usuarios',
    color='Estado',
    barmode='stack',
    title='Distribución de churned vs. no churned por plan',
    color_discrete_map={'Churned': '#FF6F61', 'No Churned': '#6BA292'},
    text_auto=True
)

fig_churn_plan.update_layout(yaxis_title='Número de usuarios')
fig_churn_plan.show()

In [ ]:
df_ = pd.read_csv("https://drive.google.com/uc?export=download&id=13b2-OSvLH_wMbgUtBD41b-JaLVsrJ3Sh")
print(df_.columns)


In [ ]:
# import gdown
# import os

# # Descarga solo si el archivo aún no existe
# if not os.path.exists("../data/df.csv"):
#     url = "https://drive.google.com/uc?id=13b2-OSvLH_wMbgUtBD41b-JaLVsrJ3Sh"
#     gdown.download(url, "../data/df.csv", quiet=False)

# df = pd.read_csv("../data/df.csv", parse_dates=["create_date_user", "create_date_transaction"])


In [ ]:
df.columns

In [ ]:
df[df['user_id'] == 'user_1303']

In [ ]:

df_transactions[df_transactions['user_id']=='user_1303']